In [42]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import minmax_scale , StandardScaler , OneHotEncoder , LabelEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [39]:
diabetes = pd.read_csv('Diabetes.csv')
adult = pd.read_csv('adult.csv')
diabetes.head()

,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
0,502,17975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,735,34221,M,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,420,47975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,680,87656,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,504,34223,M,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N


In [32]:
num_cols_diabetes = [
    'AGE', 'Urea', 'Cr', 'HbA1c', 'Chol',
    'TG', 'HDL', 'LDL', 'VLDL', 'BMI'
]

diabetes[num_cols_diabetes] = diabetes[num_cols_diabetes].replace(0, np.nan)

num_imputer = SimpleImputer(strategy='mean')
diabetes[num_cols_diabetes] = num_imputer.fit_transform(diabetes[num_cols_diabetes])
print(diabetes.isnull().sum())


ID           0
No_Pation    0
Gender       0
AGE          0
Urea         0
Cr           0
HbA1c        0
Chol         0
TG           0
HDL          0
LDL          0
VLDL         0
BMI          0
CLASS        0
dtype: int64


In [35]:
label_encoder = LabelEncoder()

diabetes['Gender']=label_encoder.fit_transform(diabetes['Gender'])
diabetes['CLASS']=label_encoder.fit_transform(diabetes['CLASS'])

print("Gender after encoding:", diabetes['Gender'].unique())
print("CLASS after encoding:", diabetes['CLASS'].unique())

Gender after encoding: [0 1 2]
CLASS after encoding: [0 1 2 3]


In [38]:
def remove_outliers_irq(df, columns):
    initial_rows = df.shape[0]
    for col in columns:
      Q1=df[col].quantile(0.25)
      Q3=df[col].quantile(0.75)
      IRQ=Q3-Q1
      df=df[(df[col] >=Q1-1.5*IRQ) & (df[col]<=Q3+1.5*IRQ)]
    final_rows = df.shape[0]
    print("\nRows removed due to outliers:", initial_rows - final_rows)
    return df
diabetes =remove_outliers_irq(diabetes, num_cols_diabetes)
print("Dataset shape after outlier removal:", diabetes.shape)


Rows removed due to outliers: 38
Dataset shape after outlier removal: (640, 14)


In [47]:
minmax = MinMaxScaler()

# Define columns to be scaled: numerical columns + encoded categorical columns
# Exclude 'ID' and 'No_Pation' as they are identifiers
columns_to_scale = [col for col in diabetes.columns if col not in ['ID', 'No_Pation']]

diabetes_scaled = diabetes[columns_to_scale]
diabetes_minmax = minmax.fit_transform(diabetes_scaled)

diabetes_minmax_df = pd.DataFrame(diabetes_minmax, columns=columns_to_scale)
display(diabetes_minmax_df.head())

,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
0,0.0,0.508475,0.109375,0.050378,0.264901,0.407767,0.044444,0.226804,0.114583,0.011461,0.173913,0.0
1,0.5,0.101695,0.104167,0.070529,0.264901,0.359223,0.081481,0.092784,0.187500,0.014327,0.139130,0.0
2,0.0,0.508475,0.109375,0.050378,0.264901,0.407767,0.044444,0.226804,0.114583,0.011461,0.173913,0.0
3,0.0,0.508475,0.109375,0.050378,0.264901,0.407767,0.044444,0.226804,0.114583,0.011461,0.173913,0.0
4,0.5,0.220339,0.171875,0.050378,0.264901,0.475728,0.051852,0.061856,0.177083,0.008596,0.069565,0.0


In [46]:
label_encoder = LabelEncoder()

# Ensure Gender and CLASS are numerically encoded before scaling
diabetes['Gender'] = label_encoder.fit_transform(diabetes['Gender'])
diabetes['CLASS'] = label_encoder.fit_transform(diabetes['CLASS'])

print("Gender unique values after re-encoding:", diabetes['Gender'].unique())
print("CLASS unique values after re-encoding:", diabetes['CLASS'].unique())

Gender unique values after re-encoding: [0 1 2]
CLASS unique values after re-encoding: [0 1 2 3 4]
